In [ ]:
import pandas as pd

In [ ]:
labels_path = "/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/notebooks/invoice_detection/invoice_page_labels.txt"


labels_df = pd.read_csv(labels_path, sep=",", header=None, names=["ticket_uuid", "invoice_page_start", "invoice_page_end"])
# remove first row
labels_df = labels_df.iloc[1:]
labels_df.reset_index(drop=True, inplace=True)

In [ ]:
labels_df

In [ ]:
## NOTE: data sources are both full raw dataset and prod dataset

In [ ]:
full_raw_dataset_path = "/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/data/raw/final_raw_data.csv"
prod_data_path = "/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/notebooks/invoice_detection/invoice_data_with_s3.csv"

In [ ]:
full_raw_dataset = pd.read_csv(full_raw_dataset_path)
prod_data = pd.read_csv(prod_data_path)

In [ ]:
from_raw = pd.merge(labels_df, full_raw_dataset, on="ticket_uuid", how="inner")
from_prod = pd.merge(labels_df, prod_data, on="ticket_uuid", how="inner")

In [ ]:
from_raw

In [ ]:
from_raw.rename(columns={"object_key": "s3_key"}, inplace=True)
from_raw['s3_bucket'] = "pair-email-classification"

In [ ]:
from_raw.columns

In [ ]:
from_prod

In [ ]:
from_prod.columns

In [ ]:
from_prod.rename(columns={"job_id": "textract_job_id"}, inplace=True)

In [ ]:
from_raw['source'] = 'raw'
from_prod['source'] = 'prod'

In [ ]:
positive_invoice_samples = pd.concat([from_raw,from_prod], ignore_index=True)
positive_invoice_samples

In [ ]:
all_colums = set(from_raw.columns).union(set(from_prod.columns))
print(all_colums)

In [ ]:
columns_to_keep = ["ticket_uuid", "attachment_id", "invoice_page_start", "invoice_page_end", "s3_key","s3_bucket",'textract_job_id','number_of_pages', 'source', 'is_ve_with_invoice', 'textract_blocks']

In [ ]:
positive_invoice_samples = positive_invoice_samples[columns_to_keep]
positive_invoice_samples

In [ ]:
print(positive_invoice_samples.attachment_id.nunique()==positive_invoice_samples.shape[0])

In [ ]:
print(positive_invoice_samples['source'].value_counts())

### Add Negative Samples

In [ ]:
## add negative samples from raw dataset where there is no invoice page
negative_samples_raw = full_raw_dataset[full_raw_dataset['is_ve_with_invoice'] == False]
negative_samples_raw

In [ ]:
negative_samples_raw.rename(columns={"object_key": "s3_key"}, inplace=True)
include = [c for c in columns_to_keep if c in negative_samples_raw.columns]
negative_samples_raw = negative_samples_raw[include]
negative_samples_raw.reset_index(drop=True, inplace=True)
negative_samples_raw['s3_bucket'] = "pair-email-classification"

In [ ]:
negative_samples_raw

In [ ]:
negative_samples_raw['source'] = 'raw'

### add negative data from prod

In [ ]:
from python_utilities.db_connection import DbConnection

analytics_db = DbConnection('ANALYTICS', 'PROD_RDS')
neg_samples_prod = analytics_db.sql_to_df("""
SELECT *
FROM (llm_attachments_predictions lpp 
LEFT JOIN llm_attachments la
ON lpp.attachment_id = la.attachment_id) 
LEFT JOIN textract_jobs tj
ON lpp.attachment_id = tj.attachment_id
WHERE lpp.subtype = 'is_invoice_inside'
AND lpp.value NOT LIKE '%%True%%'
LIMIT 150
""")

In [ ]:
neg_samples_prod

In [ ]:
# filter columns
include = [c for c in columns_to_keep if c in neg_samples_prod.columns] + ['job_id'] + ['s3_bucket']

In [ ]:
neg_samples_prod = neg_samples_prod[include]
neg_samples_prod.rename(columns={"job_id": "textract_job_id"}, inplace=True)
neg_samples_prod['s3_bucket'] = "pair-email-classification"
neg_samples_prod['source'] = 'prod'

In [ ]:
# there are 3 attachment_id columns, drop the first two
neg_samples_prod = neg_samples_prod.loc[:,~neg_samples_prod.columns.duplicated()]
# there is 2 s3_bucket columns, drop the first one
neg_samples_prod = neg_samples_prod.loc[:,~neg_samples_prod.columns.duplicated()]

In [ ]:
neg_samples_prod

In [ ]:
negative_samples_raw

In [ ]:
neg_samples_prod['is_ve_with_invoice'] = None

In [ ]:
negative_samples = pd.concat([negative_samples_raw, neg_samples_prod], ignore_index=True)

In [ ]:
negative_samples


In [ ]:
negative_samples['is_invoice_inside'] = False
positive_invoice_samples['is_invoice_inside'] = True

full_invoice_detection_dataset = pd.concat([positive_invoice_samples, negative_samples], ignore_index=True)

In [ ]:
full_invoice_detection_dataset

In [ ]:
full_invoice_detection_dataset.shape

In [ ]:
full_invoice_detection_dataset.columns

In [ ]:
#full_invoice_detection_dataset.to_csv("/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/notebooks/invoice_detection/full_invoice_detection_dataset_INPROGRESS.csv", index=False)

In [ ]:
full_invoice_detection_dataset = pd.read_csv("/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/notebooks/invoice_detection/full_invoice_detection_dataset_INPROGRESS.csv")

In [ ]:
full_invoice_detection_dataset.is_invoice_inside.value_counts()

### Fill textract blocks and number of pages fields

In [ ]:
full_invoice_detection_dataset

## 1) downlaod pdfs

In [ ]:
download_dir =  r"/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/assets/pdfs/invoice_detection_dataset"

In [ ]:
import sys
sys.path.append("/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation")
from utils.prod_utils import download_pdf_by_document_s3_info

In [ ]:
import boto3
import os
session = boto3.Session(profile_name="739275445236_DataScienceUser")
s3 = session.client("s3")


In [ ]:
full_invoice_detection_dataset

In [ ]:
print(download_dir)

In [ ]:

pdf_paths = []
for index, row in full_invoice_detection_dataset.iterrows():
    s3_key = row['s3_key']
    s3_bucket = row['s3_bucket']
    attachment_id = row['attachment_id']
    is_invoice_inside = row['is_invoice_inside']
    
    # create the local file path
    local_file_path = f"{download_dir}/positive/{attachment_id}.pdf" if is_invoice_inside else f"{download_dir}/negative/{attachment_id}.pdf"
    directory = os.path.dirname(local_file_path)
    
    # create base directory if not exists
    os.makedirs(directory, exist_ok=True)
    
    pdf_paths.append(local_file_path)
    
    if not os.path.exists(local_file_path):
        # download the file from S3
        try:
            download_pdf_by_document_s3_info(s3,s3_bucket, s3_key, directory, file_name=f"{attachment_id}.pdf")
        except Exception as e:
            s3_bucket = "pair-data-engineering-new"
            try:
                download_pdf_by_document_s3_info(s3,s3_bucket, s3_key, directory, file_name=f"{attachment_id}.pdf")
                # update the s3_bucket in the dataframe
                full_invoice_detection_dataset.at[index, 's3_bucket'] = s3_bucket
            except Exception as e:
                print(f"Failed to download {s3_key} from both buckets. Error: {e}")
                
    
    
full_invoice_detection_dataset['local_file_path'] = pdf_paths

In [ ]:
import os
for path in full_invoice_detection_dataset['local_file_path']:
    if not os.path.exists(path):
        print(f"File does not exist: {path}")

In [ ]:
#full_invoice_detection_dataset.to_csv("/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/notebooks/invoice_detection/full_invoice_detection_dataset.csv", index=False)

In [ ]:
full_invoice_detection_dataset =pd.read_csv("/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/notebooks/invoice_detection/full_invoice_detection_dataset.csv")

In [ ]:
full_invoice_detection_dataset

## 2) add textract results

In [ ]:

# example link: s3://pair-data-engineering-new/ocr_prepared_output/74100a5823898009a4d7353a183d4c3b4582a47e5178723615aef87188efe900.json
full_invoice_detection_dataset['textract_s3_link'] = full_invoice_detection_dataset.apply(lambda row: f"s3://pair-data-engineering-new/ocr_prepared_output/{row['textract_job_id']}.json" if pd.notna(row['textract_job_id']) else None, axis=1)

In [ ]:
missing_textract_blocks = full_invoice_detection_dataset[full_invoice_detection_dataset['textract_blocks'].isna()]
textract_s3_links_fetch = missing_textract_blocks['textract_s3_link'].tolist()

In [ ]:
missing_textract_blocks

In [ ]:
missing_textract_blocks

In [ ]:
missing_textract_blocks.source.value_counts()

In [ ]:
from typing import Any
def get_jsons_from_s3_parallel_custom(
    s3_links: list[str],
    max_workers: int = 20,
) -> list[dict[str, Any]]:
    """Download Textract JSONs from S3 in parallel using a thread pool.

    Returns results in the same order as ``s3_links``.
    Failed downloads return a fallback empty-line object.
    """
    import boto3
    from botocore.exceptions import ClientError
    from concurrent.futures import ThreadPoolExecutor, as_completed
    import json
    import logging

    logger = logging.getLogger(__name__)
    session = boto3.Session(profile_name="739275445236_DataScienceUser")
    s3_client = session.client("s3")
    fallback = [{"BlockType": "LINE", "Text": ""}]

    def _download_one(link: str) -> dict[str, Any]:
        try:
            parts = link.replace("s3://", "").split("/")
            bucket_name = parts[0]
            key = "/".join(parts[1:])
            response = s3_client.get_object(Bucket=bucket_name, Key=key)
            content = response["Body"].read().decode("utf-8")
            return json.loads(content)
        except ClientError as e:
            logger.error(f"Error retrieving object from {link}: {e}")
            return fallback
        except json.JSONDecodeError:
            logger.error(f"Error decoding JSON from {link}")
            return fallback
        except Exception as e:
            logger.error(f"Error from {link} {e}")
            return fallback

    results: list[dict[str, Any] | None] = [None] * len(s3_links)
    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        futures = {
            pool.submit(_download_one, link): idx
            for idx, link in enumerate(s3_links)
        }
        for future in as_completed(futures):
            results[futures[future]] = future.result()
    return results

In [ ]:
textract_jsons = get_jsons_from_s3_parallel_custom(textract_s3_links_fetch)

In [ ]:
# Persist successfully fetched textract JSONs into the dataframe
import json

fallback_marker = [{"BlockType": "LINE", "Text": ""}]
full_invoice_detection_dataset["textract_blocks"] = full_invoice_detection_dataset["textract_blocks"].astype(object)

success_count = 0
for local_idx, blocks in enumerate(textract_jsons):
    if blocks is None or blocks == fallback_marker:
        continue
    df_idx = missing_textract_blocks.index[local_idx]
    full_invoice_detection_dataset.at[df_idx, "textract_blocks"] = json.dumps(blocks)
    success_count += 1

print(f"Wrote textract_blocks for {success_count} rows")
print(
    "Remaining rows with missing textract_blocks:",
    int(full_invoice_detection_dataset["textract_blocks"].isna().sum()),
)

In [ ]:
textract_jsons

### Resubmit Textract jobs for missing/failed S3 results

Some `textract_s3_link` JSONs do not exist in S3 (NoSuchKey). For those rows, upload the local PDF and submit a fresh Textract job, then update `textract_job_id`, `textract_s3_link`, and `textract_blocks`.

In [ ]:
# Detect failed downloads (the helper returns the fallback object on failure)
fallback_marker = [{"BlockType": "LINE", "Text": ""}]

failed_local_idx = [
    i for i, out in enumerate(textract_jsons) if out == fallback_marker
]
# Map back to indices in the full dataframe
failed_df_idx = missing_textract_blocks.index[failed_local_idx].tolist()
print(f"{len(failed_df_idx)} rows need a fresh Textract job out of {len(textract_jsons)}")


In [ ]:
import sys
sys.path.append("/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation")

In [ ]:
import os
import boto3
from utils.use_textract_utils import (
    _get_textract_client,
    ModelConfig,
    wait_for_job_completion,
)

session = boto3.Session(region_name="eu-central-1", profile_name="739275445236_DataScienceUser")
upload_s3_client = session.client("s3")
textract_client = _get_textract_client()

upload_bucket = ModelConfig.ocr_s3_bucket  # textract-readable bucket
upload_prefix = "ocr_source_files/invoice_detection_resubmit"


In [ ]:
# Upload local PDFs and submit Textract jobs for all failed rows first (so they run in parallel)
new_job_infos = {}  # df_idx -> job_info dict

for df_idx in failed_df_idx:
    row = full_invoice_detection_dataset.loc[df_idx]
    local_path = row["local_file_path"]
    attachment_id = row["attachment_id"]

    if not isinstance(local_path, str) or not os.path.exists(local_path):
        print(f"[skip] missing local file for idx {df_idx} ({attachment_id}): {local_path}")
        new_job_infos[df_idx] = {"job_id": None, "doc_key": None, "status": "FAILED", "error": "local file missing"}
        continue

    object_key = f"{upload_prefix}/{attachment_id}.pdf"
    try:
        upload_s3_client.upload_file(local_path, upload_bucket, object_key)
        job_id = textract_client.submit_textract_job(bucket_name=upload_bucket, document_key=object_key)
        status = "SUBMITTED" if job_id else "FAILED"
        new_job_infos[df_idx] = {"job_id": job_id, "doc_key": object_key, "status": status}
        print(f"[{status}] idx {df_idx} -> job_id {job_id}")
    except Exception as e:
        print(f"[error] idx {df_idx}: {e}")
        new_job_infos[df_idx] = {"job_id": None, "doc_key": object_key, "status": "FAILED", "error": str(e)}


In [ ]:
# Update textract_job_id and textract_s3_link immediately for submitted jobs
for df_idx, info in new_job_infos.items():
    if info.get("job_id"):
        full_invoice_detection_dataset.at[df_idx, "textract_job_id"] = info["job_id"]
        full_invoice_detection_dataset.at[df_idx, "textract_s3_link"] = (
            f"s3://pair-data-engineering-new/ocr_prepared_output/{info['job_id']}.json"
        )
print(
    "Updated textract_job_id / textract_s3_link for",
    sum(1 for v in new_job_infos.values() if v.get("job_id")),
    "rows",
)


In [ ]:
# Wait for each submitted job to complete and collect the resulting blocks
import json

new_blocks_by_idx = {}  # df_idx -> blocks list (or None)

for df_idx, info in new_job_infos.items():
    if not info.get("job_id"):
        new_blocks_by_idx[df_idx] = None
        continue
    result = wait_for_job_completion(textract_client, info, max_wait_time=600)
    if result["status"] == "SUCCEEDED":
        new_blocks_by_idx[df_idx] = result["result"]
        print(f"[ok] idx {df_idx}: {len(result['result'])} blocks")
    else:
        new_blocks_by_idx[df_idx] = None
        print(f"[fail] idx {df_idx}: {result.get('error')}")


In [ ]:
# Write blocks back into the dataframe (serialised as JSON for CSV compatibility)
full_invoice_detection_dataset["textract_blocks"] = full_invoice_detection_dataset["textract_blocks"].astype(object)

updated = 0
for df_idx, blocks in new_blocks_by_idx.items():
    if blocks is None:
        continue
    full_invoice_detection_dataset.at[df_idx, "textract_blocks"] = json.dumps(blocks)
    updated += 1

print(f"Wrote textract_blocks for {updated} rows")
print(
    "Remaining rows with missing textract_blocks:",
    int(full_invoice_detection_dataset["textract_blocks"].isna().sum()),
)


In [ ]:
full_invoice_detection_dataset

In [ ]:
full_invoice_detection_dataset.to_csv("/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/notebooks/invoice_detection/full_invoice_detection_dataset_FINAL.csv", index=False)

In [ ]:
import pandas as pd
full_invoice_detection_dataset = pd.read_csv("/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/notebooks/invoice_detection/full_invoice_detection_dataset_FINAL.csv")


In [ ]:
new_invoice = '/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/notebooks/invoice_detection/new_invoice.txt'
not_invoice_keep = '/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/notebooks/invoice_detection/not_invoice_keep.txt'

In [ ]:
new_invoice_labels = pd.read_csv(new_invoice, header=None, names=["attachment_id", "invoice_page_start","invoice_page_end"])
not_invoice_keep_labels = pd.read_csv(not_invoice_keep, header=None, names=["attachment_id"])

In [ ]:
new_invoice_labels

In [ ]:
full_invoice_detection_dataset.is_invoice_inside.value_counts()

In [ ]:
new_invoice_labels

In [ ]:
ids = new_invoice_labels['attachment_id'].tolist()
check = full_invoice_detection_dataset[full_invoice_detection_dataset['attachment_id'].isin(ids)]
check

In [ ]:
for a_id, start, end in new_invoice_labels.itertuples(index=False):
    # update dataset rows with matching attachment_id
    mask = full_invoice_detection_dataset['attachment_id'] == a_id
    full_invoice_detection_dataset.loc[mask, 'invoice_page_start'] = start
    full_invoice_detection_dataset.loc[mask, 'invoice_page_end'] = end
    full_invoice_detection_dataset.loc[mask, 'is_invoice_inside'] = True
    if  full_invoice_detection_dataset.loc[mask, 'is_ve_with_invoice'].values[0] == False:
        full_invoice_detection_dataset.loc[mask, 'is_ve_with_invoice'] = True

In [ ]:
full_invoice_detection_dataset.is_invoice_inside.value_counts()

In [ ]:
ids = new_invoice_labels['attachment_id'].tolist()
check = full_invoice_detection_dataset[full_invoice_detection_dataset['attachment_id'].isin(ids)]
check

In [ ]:
print(full_invoice_detection_dataset.head(n=2))

In [ ]:
full_invoice_detection_dataset

In [ ]:
# parse textract_blocks column from string back to list of dicts
from ast import literal_eval

full_invoice_detection_dataset["textract_blocks"] = full_invoice_detection_dataset["textract_blocks"].apply(
    lambda x: literal_eval(x) if pd.notna(x) else None
)

In [ ]:
full_invoice_detection_dataset['number_of_pages'] = full_invoice_detection_dataset['textract_blocks'].apply(lambda blocks: len(set(block['Page'] for block in blocks)))

In [ ]:
full_invoice_detection_dataset.number_of_pages.isna().sum()

In [ ]:
full_invoice_detection_dataset.number_of_pages.value_counts()

In [ ]:
invoice_inside = full_invoice_detection_dataset[full_invoice_detection_dataset['is_invoice_inside'] == True]
invoice_inside.number_of_pages.value_counts().sort_index()

In [ ]:
full_invoice_detection_dataset

In [ ]:
check = full_invoice_detection_dataset.iloc[0]['textract_blocks']

In [ ]:
from collections import defaultdict

pages_to_text_dict = defaultdict(list[str])
for block in check:
    if block['BlockType'] == "LINE":
        page_number = block['Page']
        text = block["Text"]
        pages_to_text_dict[page_number].append(text)
        
for page_number, lines in pages_to_text_dict.items():
    pages_to_text_dict[page_number] = "\n".join(lines)

In [ ]:
def create_page_to_text_dict(textract_block: list[dict]) -> dict[int, str]:
    pages_to_text_dict = defaultdict(list[str])
    for block in textract_block:
        if block['BlockType'] == "LINE":
            page_number = block['Page']
            text = block["Text"]
            pages_to_text_dict[page_number].append(text)

    for page_number, lines in pages_to_text_dict.items():
        pages_to_text_dict[page_number] = "\n".join(lines)
        
    return dict(pages_to_text_dict)

full_invoice_detection_dataset['pages_to_text_dict'] = full_invoice_detection_dataset['textract_blocks'].apply(lambda blocks: create_page_to_text_dict(blocks) if isinstance(blocks, list) else None)

In [ ]:
full_invoice_detection_dataset

In [ ]:
def create_text_with_page_markers(pages_to_text_dict: dict):
    for page, text in pages_to_text_dict.items():
        page = int(page)
        text = f"<page_{page}>\n{text}"
        pages_to_text_dict[page] = text
    full_text = "\n".join(pages_to_text_dict.values())
    return full_text




In [ ]:
import sys
sys.path.append("/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation")

In [ ]:
from intent_recognition.src.domain.base.blueprints import AfterCourtPreprocessingBlueprint
from utils.intent_recog_utils import apply_text_cleaning

config = AfterCourtPreprocessingBlueprint(
    clean_text_type="preprocessed",
    normalize_whitespace=True,
    remove_short_lines=False,   # keep short lines
    short_line_threshold=3,
    remove_html_tags=False,
    lowercase=True     
)


In [ ]:
import pandas as pd
full_invoice_detection_dataset = pd.read_csv("/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/notebooks/invoice_detection/full_invoice_detection_dataset_WITH_PAGES_TO_TEXT_FINAL.csv")

In [ ]:
test = full_invoice_detection_dataset.iloc[0]
test['pages_to_text_dict'] = eval(test['pages_to_text_dict'])
text = create_text_with_page_markers(test['pages_to_text_dict'])
cleaned_text = apply_text_cleaning(text, config=config)
print(cleaned_text)


In [ ]:
def process_row(pages_to_text_dict_str):
    if pd.isna(pages_to_text_dict_str):
        return None
    pages_to_text_dict = eval(pages_to_text_dict_str)
    text = create_text_with_page_markers(pages_to_text_dict)
    return apply_text_cleaning(text, config=config)

full_invoice_detection_dataset['cleaned_text'] = full_invoice_detection_dataset['pages_to_text_dict'].apply(process_row)

In [ ]:
print(full_invoice_detection_dataset['cleaned_text'].iloc[2])

In [ ]:
full_invoice_detection_dataset.to_csv("/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/notebooks/invoice_detection/full_invoice_detection_dataset_WITH_PAGES_TO_TEXT_FINAL.csv", index=False)

In [ ]:
import pandas as pd

full_invoice_detection_dataset = pd.read_csv("/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/notebooks/invoice_detection/full_invoice_detection_dataset_WITH_PAGES_TO_TEXT_FINAL.csv")

In [ ]:
full_invoice_detection_dataset.columns

In [ ]:
full_invoice_detection_dataset.drop(columns=['textract_blocks','pages_to_text_dict'], inplace=True)

full_invoice_detection_dataset.to_csv("/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/notebooks/invoice_detection/full_invoice_detection_dataset_FINAL_LIGHT.csv", index=False)